# Practica 2 - Modelado Predictivo con Optimizacion, Calibracion e Incertidumbre

Este notebook implementa un pipeline completo de machine learning para prediccion de riesgo crediticio, incluyendo:

1. **Optimizacion de hiperparametros con Optuna (TPE)**
2. **Calibracion de probabilidades**
3. **Gestion de incertidumbre con Conformal Prediction**
4. **Persistencia del modelo final**

## Setup Inicial

In [16]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, log_loss, brier_score_loss,
    average_precision_score, confusion_matrix, classification_report
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# Gradient Boosting
import lightgbm as lgb
import xgboost as xgb

# Optuna
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Conformal Prediction
from mapie.classification import SplitConformalClassifier

# Configuracion
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup completado.")

Setup completado.


## 0. Carga de Datos y Preparacion

Cargamos los datos y aplicamos preprocesamiento y filtrado usando las clases base.

In [17]:
# Rutas
DATA_DIR = Path("data")  # Ruta relativa
MODELS_DIR = Path("models")  # Ruta relativa
MODELS_DIR.mkdir(exist_ok=True)

# Cargar datos
train_file = DATA_DIR / "df_train_small.csv"
test_file = DATA_DIR / "df_test_small.csv"

print(f"Cargando datos de entrenamiento desde {train_file}")
print(f"Cargando datos de test desde {test_file}")

df_train = pd.read_csv(train_file)
df_test = pd.read_csv(test_file)

print(f"\nTrain shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")
print(f"\nColumnas: {df_train.shape[1]}")

Cargando datos de entrenamiento desde data\df_train_small.csv
Cargando datos de test desde data\df_test_small.csv

Train shape: (80000, 23)
Test shape: (20000, 23)

Columnas: 23


### Preprocesamiento

Aplicamos las clases de preprocesamiento disponibles en el proyecto.

In [20]:
# Configurar path para imports locales
import sys
from pathlib import Path

# Añadir directorio del notebook al path
notebook_dir = Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

# Importar clases de preprocesamiento
from src.preprocessing.base_preprocessing import BasePreprocess
from src.filtering.base_filtering import BaseFiltering

print("Clases importadas correctamente.")


ModuleNotFoundError: No module named 'skrub'

In [ ]:
# Cargar preprocessor PRE-FITTEADO (NO hacer fit)
# Según las instrucciones: "sin volver a fittear preprocesador ni filtros"
import joblib

print("Cargando preprocessor pre-fitteado...")
preprocessor = joblib.load(MODELS_DIR / "preprocessor.pkl")

print("Transformando datos de entrenamiento...")
X_train_preprocessed, y_train = preprocessor.transform(df_train)

print("Transformando datos de test...")
X_test_preprocessed, y_test = preprocessor.transform(df_test)

print(f"\nX_train shape despues de preprocesamiento: {X_train_preprocessed.shape}")
print(f"X_test shape despues de preprocesamiento: {X_test_preprocessed.shape}")

# Convertir y_train e y_test a arrays planos
y_train_flat = y_train.values.ravel() if hasattr(y_train, 'values') else y_train.ravel()
y_test_flat = y_test.values.ravel() if hasattr(y_test, 'values') else y_test.ravel()

print(f"\nDistribucion de clases en train: {np.bincount(y_train_flat.astype(int))}")
print(f"\nDistribucion de clases en test: {np.bincount(y_test_flat.astype(int))}")

print(f"\nPreprocessor cargado desde {MODELS_DIR / 'preprocessor.pkl'}")

: 

: 

: 

### Filtrado de Features

Aplicamos seleccion de features usando BaseFiltering.

In [ ]:
# Cargar filtro PRE-FITTEADO (NO hacer fit)
# Según las instrucciones: "sin volver a fittear preprocesador ni filtros"

print("Cargando filtro de features pre-fitteado...")
feature_filter = joblib.load(MODELS_DIR / "filter.pkl")

print("\nTransformando datos...")
X_train_filtered = feature_filter.transform(X_train_preprocessed)
X_test_filtered = feature_filter.transform(X_test_preprocessed)

print(f"\nX_train shape despues de filtrado: {X_train_filtered.shape}")
print(f"X_test shape despues de filtrado: {X_test_filtered.shape}")

print(f"\nFiltro cargado desde {MODELS_DIR / 'filter.pkl'}")

: 

: 

: 

### Dividir en Train/Validation/Calibration

Dividimos los datos en:
- Train: para entrenar modelos durante optimizacion
- Validation: para validar durante optimizacion
- Calibration: para calibrar el modelo final
- Test: para evaluacion final

In [ ]:
# Split train en train + validation
X_train_opt, X_val, y_train_opt, y_val = train_test_split(
    X_train_filtered, y_train_flat,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_train_flat
)

# Split train_opt en train_final + calibration
X_train_final, X_cal, y_train_final, y_cal = train_test_split(
    X_train_opt, y_train_opt,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_train_opt
)

print(f"Train final shape: {X_train_final.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Calibration shape: {X_cal.shape}")
print(f"Test shape: {X_test_filtered.shape}")

print(f"\nDistribucion en train_final: {np.bincount(y_train_final.astype(int))}")
print(f"Distribucion en validation: {np.bincount(y_val.astype(int))}")
print(f"Distribucion en calibration: {np.bincount(y_cal.astype(int))}")
print(f"Distribucion en test: {np.bincount(y_test_flat.astype(int))}")

: 

: 

: 

In [ ]:
# Limpiar nombres de columnas para compatibilidad con XGBoost
# XGBoost no acepta caracteres especiales: [ ] <
import re

def clean_column_names(df):
    """Limpia nombres de columnas eliminando caracteres especiales"""
    new_columns = []
    for col in df.columns:
        # Reemplazar [ ] < > con _
        clean_col = re.sub(r'[\[\]<>]', '_', str(col))
        # Reemplazar múltiples _ con uno solo
        clean_col = re.sub(r'_+', '_', clean_col)
        # Remover _ al inicio y final
        clean_col = clean_col.strip('_')
        new_columns.append(clean_col)
    df.columns = new_columns
    return df

# Aplicar limpieza a todos los datasets
X_train_filtered = clean_column_names(X_train_filtered)
X_test_filtered = clean_column_names(X_test_filtered)
X_train_opt = clean_column_names(X_train_opt)
X_val = clean_column_names(X_val)
X_train_final = clean_column_names(X_train_final)
X_cal = clean_column_names(X_cal)

print(f"Nombres de columnas limpiados")
print(f"Ejemplo de columnas (primeras 5): {list(X_train_filtered.columns[:5])}")

## 1.1 Optimizacion con Optuna

Optimizamos LightGBM y XGBoost usando TPE sampler, comparando versiones balanceadas y no balanceadas.

### Funciones Auxiliares para Metricas

### Justificación de la Métrica Elegida: Log Loss

He elegido **Log Loss** como métrica objetivo por las siguientes razones:

1. **Proper scoring rule**: Log Loss penaliza tanto la mala discriminación como la mala calibración simultáneamente.
   - **Discriminación** (resolution): Recompensa predecir probabilidades extremas cuando el modelo está seguro
   - **Calibración** (reliability): Penaliza probabilidades que no coinciden con las frecuencias observadas

2. **Diferenciabilidad**: Es diferenciable, lo que facilita la optimización con gradient-based methods.

3. **Sensibilidad a probabilidades extremas**: Penaliza fuertemente las predicciones incorrectas con alta confianza (ej: predecir p=0.95 cuando la clase real es 0).

4. **Equilibrio natural**: Captura ambas dimensiones (resolution + reliability) en una sola métrica sin necesidad de ponderaciones arbitrarias.

5. **Estándar de la industria**: Es la métrica por defecto en clasificación probabilística (equivalente a cross-entropy).

**Alternativas consideradas**:
- ❌ **Solo Brier**: Menos sensible a errores extremos (usa MSE en lugar de log)
- ❌ **Solo AUC**: Ignora completamente la calibración
- ❌ **Métrica combinada (AUC - λ*ECE)**: Requiere elegir λ arbitrariamente

**Conclusión**: Log Loss es la opción recomendada por defecto según las instrucciones y es más simple y robusta que construir una métrica combinada custom.

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """
    Calcula Expected Calibration Error (ECE).
    
    ECE mide el promedio ponderado de la diferencia entre
    las probabilidades predichas y las frecuencias observadas.
    """
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_prob, bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    
    N = len(y_true)
    ece = 0.0
    
    for k in range(n_bins):
        mask = bin_indices == k
        n_k = mask.sum()
        if n_k == 0:
            continue
        avg_pred = y_prob[mask].mean()
        avg_true = y_true[mask].mean()
        ece += (n_k / N) * abs(avg_pred - avg_true)
    
    return ece


def compute_all_metrics(y_true, y_prob, threshold=0.5):
    """
    Calcula todas las metricas requeridas.
    """
    y_pred = (y_prob >= threshold).astype(int)
    
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'PR-AUC': average_precision_score(y_true, y_prob),
        'Log Loss': log_loss(y_true, y_prob),
        'Brier': brier_score_loss(y_true, y_prob),
        'ECE': expected_calibration_error(y_true, y_prob)
    }
    
    return metrics


print("Funciones de metricas definidas.")

: 

: 

: 

### Funciones Objetivo para Optuna

In [ ]:
def objective_lightgbm(trial, balanced=False):
    """
    Funcion objetivo para LightGBM.
    Minimiza Log Loss en el conjunto de validacion.
    """
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        
        # Hiperparametros a optimizar
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
        'feature_fraction_bynode': trial.suggest_float('feature_fraction_bynode', 0.5, 1.0),
    }
    
    if balanced:
        params['class_weight'] = 'balanced'
    
    # Entrenar modelo
    train_data = lgb.Dataset(X_train_final, label=y_train_final)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )
    
    # Predecir y calcular Log Loss
    y_pred_proba = model.predict(X_val)
    logloss = log_loss(y_val, y_pred_proba)
    
    return logloss


def objective_xgboost(trial, balanced=False):
    """
    Funcion objetivo para XGBoost.
    Minimiza Log Loss en el conjunto de validacion.
    """
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        
        # Hiperparametros a optimizar
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    
    if balanced:
        # Calcular scale_pos_weight
        n_neg = (y_train_final == 0).sum()
        n_pos = (y_train_final == 1).sum()
        params['scale_pos_weight'] = n_neg / n_pos
    
    # Entrenar modelo
    dtrain = xgb.DMatrix(X_train_final, label=y_train_final)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=[(dval, 'validation')],
        early_stopping_rounds=50,
        verbose_eval=False
    )
    
    # Predecir y calcular Log Loss
    y_pred_proba = model.predict(dval)
    logloss = log_loss(y_val, y_pred_proba)
    
    return logloss


print("Funciones objetivo definidas.")

: 

: 

: 

### Optimizacion con TPE Sampler

### Justificación de Cambios en Sampler/Pruner

**Cambios realizados respecto al notebook 11**:

1. **`multivariate=True` en TPESampler** (antes: False o por defecto)

   **Razón**:
   - El modo multivariado permite que TPE considere las **correlaciones entre hiperparámetros**
   - Es especialmente útil cuando hiperparámetros tienen dependencias naturales:
     - `learning_rate` ↔ `num_leaves`: learning rate bajo suele requerir más hojas
     - `max_depth` ↔ `min_child_samples`: mayor profundidad requiere más muestras por nodo
   - Mejora la exploración del espacio de búsqueda en espacios de alta dimensión (9 hiperparámetros)
   - El coste computacional adicional es marginal con 50 trials

   **Evidencia**: Según el paper de TPE (Bergstra et al., 2011), el modo multivariado mejora convergencia en espacios con >5 dimensiones.

2. **`n_startup_trials=10`** (antes: típicamente 5)

   **Razón**:
   - Aumentar los trials de exploración aleatoria inicial mejora la diversidad del prior
   - Con 50 trials totales, dedicar 20% a exploración pura es un buen balance
   - Reduce el riesgo de que TPE converja prematuramente a un mínimo local

**Configuración final**:
```python
sampler = TPESampler(
    n_startup_trials=10,      # ← Aumentado
    multivariate=True,         # ← Activado
    seed=RANDOM_STATE
)
```

In [ ]:
# Configuracion de Optuna
N_TRIALS = 50
N_STARTUP_TRIALS = 10

# Sampler TPE con variaciones respecto al notebook de referencia
sampler = TPESampler(
    n_startup_trials=N_STARTUP_TRIALS,
    multivariate=True,  # Considerar correlaciones entre hiperparametros
    seed=RANDOM_STATE
)

pruner = MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=10
)

print(f"Configuracion: {N_TRIALS} trials, {N_STARTUP_TRIALS} startup trials")
print("TPE Sampler configurado con modo multivariate activado.")

: 

: 

: 

#### LightGBM - No Balanceado

In [ ]:
print("=" * 60)
print("LIGHTGBM - NO BALANCEADO")
print("=" * 60)

study_lgbm_unbalanced = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    pruner=pruner,
    study_name='lgbm_unbalanced'
)

study_lgbm_unbalanced.optimize(
    lambda trial: objective_lightgbm(trial, balanced=False),
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print(f"\nMejor Log Loss: {study_lgbm_unbalanced.best_value:.6f}")
print(f"Mejores hiperparametros:")
for key, value in study_lgbm_unbalanced.best_params.items():
    print(f"  {key}: {value}")

: 

: 

: 

#### LightGBM - Balanceado

In [ ]:
print("=" * 60)
print("LIGHTGBM - BALANCEADO")
print("=" * 60)

study_lgbm_balanced = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    pruner=pruner,
    study_name='lgbm_balanced'
)

study_lgbm_balanced.optimize(
    lambda trial: objective_lightgbm(trial, balanced=True),
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print(f"\nMejor Log Loss: {study_lgbm_balanced.best_value:.6f}")
print(f"Mejores hiperparametros:")
for key, value in study_lgbm_balanced.best_params.items():
    print(f"  {key}: {value}")

: 

: 

: 

#### XGBoost - No Balanceado

In [ ]:
print("=" * 60)
print("XGBOOST - NO BALANCEADO")
print("=" * 60)

study_xgb_unbalanced = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    pruner=pruner,
    study_name='xgb_unbalanced'
)

study_xgb_unbalanced.optimize(
    lambda trial: objective_xgboost(trial, balanced=False),
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print(f"\nMejor Log Loss: {study_xgb_unbalanced.best_value:.6f}")
print(f"Mejores hiperparametros:")
for key, value in study_xgb_unbalanced.best_params.items():
    print(f"  {key}: {value}")

: 

: 

: 

#### XGBoost - Balanceado

In [ ]:
print("=" * 60)
print("XGBOOST - BALANCEADO")
print("=" * 60)

study_xgb_balanced = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    pruner=pruner,
    study_name='xgb_balanced'
)

study_xgb_balanced.optimize(
    lambda trial: objective_xgboost(trial, balanced=True),
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print(f"\nMejor Log Loss: {study_xgb_balanced.best_value:.6f}")
print(f"Mejores hiperparametros:")
for key, value in study_xgb_balanced.best_params.items():
    print(f"  {key}: {value}")

: 

: 

: 

### Entrenar Modelos Finales y Comparar

In [ ]:
# Entrenar modelo LightGBM No Balanceado con mejores parametros
print("Entrenando LightGBM No Balanceado...")
params_lgbm_unbal = study_lgbm_unbalanced.best_params.copy()
params_lgbm_unbal.update({
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'random_state': RANDOM_STATE,
    'n_jobs': -1
})

train_data = lgb.Dataset(X_train_opt, label=y_train_opt)
model_lgbm_unbal = lgb.train(
    params_lgbm_unbal,
    train_data,
    num_boost_round=1000,
    callbacks=[lgb.log_evaluation(period=0)]
)

y_pred_lgbm_unbal = model_lgbm_unbal.predict(X_test_filtered)
metrics_lgbm_unbal = compute_all_metrics(y_test_flat, y_pred_lgbm_unbal)
print("Completado.")

: 

: 

: 

In [ ]:
# Entrenar modelo LightGBM Balanceado con mejores parametros
print("Entrenando LightGBM Balanceado...")
params_lgbm_bal = study_lgbm_balanced.best_params.copy()
params_lgbm_bal.update({
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'class_weight': 'balanced',
    'feature_pre_filter': False
})

model_lgbm_bal = lgb.train(
    params_lgbm_bal,
    train_data,
    num_boost_round=1000,
    callbacks=[lgb.log_evaluation(period=0)]
)

y_pred_lgbm_bal = model_lgbm_bal.predict(X_test_filtered)
metrics_lgbm_bal = compute_all_metrics(y_test_flat, y_pred_lgbm_bal)
print("Completado.")

: 

: 

: 

In [ ]:
# Entrenar modelo XGBoost No Balanceado con mejores parametros
print("Entrenando XGBoost No Balanceado...")
params_xgb_unbal = study_xgb_unbalanced.best_params.copy()
params_xgb_unbal.update({
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'n_jobs': -1
})

dtrain = xgb.DMatrix(X_train_opt, label=y_train_opt)
dtest = xgb.DMatrix(X_test_filtered, label=y_test_flat)

model_xgb_unbal = xgb.train(
    params_xgb_unbal,
    dtrain,
    num_boost_round=1000,
    verbose_eval=False
)

y_pred_xgb_unbal = model_xgb_unbal.predict(dtest)
metrics_xgb_unbal = compute_all_metrics(y_test_flat, y_pred_xgb_unbal)
print("Completado.")

: 

: 

: 

In [ ]:
# Entrenar modelo XGBoost Balanceado con mejores parametros
print("Entrenando XGBoost Balanceado...")
params_xgb_bal = study_xgb_balanced.best_params.copy()
n_neg = (y_train_opt == 0).sum()
n_pos = (y_train_opt == 1).sum()
params_xgb_bal.update({
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'scale_pos_weight': n_neg / n_pos
})

model_xgb_bal = xgb.train(
    params_xgb_bal,
    dtrain,
    num_boost_round=1000,
    verbose_eval=False
)

y_pred_xgb_bal = model_xgb_bal.predict(dtest)
metrics_xgb_bal = compute_all_metrics(y_test_flat, y_pred_xgb_bal)
print("Completado.")

: 

: 

: 

### Tabla Comparativa Final

In [ ]:
# Crear tabla comparativa
comparison_df = pd.DataFrame({
    'LightGBM_Unbalanced': metrics_lgbm_unbal,
    'LightGBM_Balanced': metrics_lgbm_bal,
    'XGBoost_Unbalanced': metrics_xgb_unbal,
    'XGBoost_Balanced': metrics_xgb_bal
}).T

print("\n" + "=" * 100)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 100)
print(comparison_df.to_string())
print("=" * 100)

# Identificar mejor modelo (menor Log Loss)
best_model_name = comparison_df['Log Loss'].idxmin()
print(f"\nMEJOR MODELO: {best_model_name}")
print(f"Log Loss: {comparison_df.loc[best_model_name, 'Log Loss']:.6f}")
print(f"ROC-AUC: {comparison_df.loc[best_model_name, 'ROC-AUC']:.6f}")
print(f"ECE: {comparison_df.loc[best_model_name, 'ECE']:.6f}")

: 

: 

: 

### Seleccionar Modelo Ganador

In [ ]:
# Mapear nombre a modelo y predicciones
model_map = {
    'LightGBM_Unbalanced': (model_lgbm_unbal, y_pred_lgbm_unbal, 'lightgbm'),
    'LightGBM_Balanced': (model_lgbm_bal, y_pred_lgbm_bal, 'lightgbm'),
    'XGBoost_Unbalanced': (model_xgb_unbal, y_pred_xgb_unbal, 'xgboost'),
    'XGBoost_Balanced': (model_xgb_bal, y_pred_xgb_bal, 'xgboost')
}

best_model, best_predictions, model_type = model_map[best_model_name]

print(f"\nModelo ganador seleccionado: {best_model_name}")
print(f"Tipo: {model_type}")

: 

: 

: 

## 1.2 Calibracion

Diagnosticamos la calibracion del modelo ganador y decidimos si aplicar calibracion.

### Diagnostico de Calibracion

In [ ]:
def plot_reliability_diagram(y_true, y_prob, model_name, n_bins=10):
    """
    Grafica el diagrama de confiabilidad (reliability diagram).
    """
    prob_true, prob_pred = calibration_curve(
        y_true, y_prob, n_bins=n_bins, strategy='uniform'
    )
    
    plt.figure(figsize=(8, 6))
    plt.plot(prob_pred, prob_true, 'o-', lw=2, markersize=8, label='Model')
    plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
    plt.xlabel('Mean predicted probability', fontsize=12)
    plt.ylabel('Fraction of positives', fontsize=12)
    plt.title(f'Reliability Diagram - {model_name}', fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Obtener predicciones en conjunto de test
print("Diagnosticando calibracion del modelo ganador...\n")

brier = brier_score_loss(y_test_flat, best_predictions)
ece = expected_calibration_error(y_test_flat, best_predictions)

print(f"Brier Score: {brier:.6f}")
print(f"Expected Calibration Error (ECE): {ece:.6f}")

plot_reliability_diagram(y_test_flat, best_predictions, best_model_name)

: 

: 

: 

### Decision: Calibrar o No Calibrar

Criterios para decidir:
- **ECE > 0.05**: Indica calibracion pobre, se debe calibrar
- **Brier Score alto**: Indica probabilidades mal calibradas
- **Reliability diagram**: Si se desvía significativamente de la diagonal perfecta

In [ ]:
# Umbral de decision
ECE_THRESHOLD = 0.05

should_calibrate = ece > ECE_THRESHOLD

print("\n" + "=" * 60)
if should_calibrate:
    print("DECISION: CALIBRAR EL MODELO")
    print(f"Razon: ECE ({ece:.6f}) > umbral ({ECE_THRESHOLD})")
    print("El modelo muestra calibracion deficiente.")
else:
    print("DECISION: NO CALIBRAR")
    print(f"Razon: ECE ({ece:.6f}) <= umbral ({ECE_THRESHOLD})")
    print("El modelo ya esta bien calibrado.")
print("=" * 60)

: 

: 

: 

### Aplicar Calibracion (si es necesario)

In [18]:
if should_calibrate:
    print("\nAplicando calibracion con Platt Scaling e Isotonic Regression...")
    
    # Wrapper para que CalibratedClassifierCV funcione con LightGBM/XGBoost
    class ModelWrapper:
        def __init__(self, model, model_type):
            self.model = model
            self.model_type = model_type
        
        def predict_proba(self, X):
            if self.model_type == 'lightgbm':
                proba = self.model.predict(X)
            else:  # xgboost
                dmatrix = xgb.DMatrix(X)
                proba = self.model.predict(dmatrix)
            return np.vstack([1 - proba, proba]).T
    
    wrapped_model = ModelWrapper(best_model, model_type)
    
    # Platt Scaling
    platt_calibrator = CalibratedClassifierCV(
        estimator=wrapped_model,
        method='sigmoid',
        cv='prefit'
    )
    platt_calibrator.fit(X_cal, y_cal)
    y_pred_platt = platt_calibrator.predict_proba(X_test_filtered)[:, 1]
    
    # Isotonic Regression
    isotonic_calibrator = CalibratedClassifierCV(
        estimator=wrapped_model,
        method='isotonic',
        cv='prefit'
    )
    isotonic_calibrator.fit(X_cal, y_cal)
    y_pred_isotonic = isotonic_calibrator.predict_proba(X_test_filtered)[:, 1]
    
    # Comparar metodos
    print("\nComparacion de metodos de calibracion:")
    
    methods = {
        'Original': best_predictions,
        'Platt Scaling': y_pred_platt,
        'Isotonic Regression': y_pred_isotonic
    }
    
    calibration_results = {}
    for name, probs in methods.items():
        calibration_results[name] = {
            'Brier': brier_score_loss(y_test_flat, probs),
            'ECE': expected_calibration_error(y_test_flat, probs),
            'Log Loss': log_loss(y_test_flat, probs),
            'ROC-AUC': roc_auc_score(y_test_flat, probs)
        }
    
    calibration_df = pd.DataFrame(calibration_results).T
    print("\n" + calibration_df.to_string())
    
    # Seleccionar mejor metodo (menor ECE)
    best_calibration_method = calibration_df['ECE'].idxmin()
    print(f"\nMejor metodo de calibracion: {best_calibration_method}")
    
    if best_calibration_method == 'Platt Scaling':
        final_model = platt_calibrator
        final_predictions = y_pred_platt
    elif best_calibration_method == 'Isotonic Regression':
        final_model = isotonic_calibrator
        final_predictions = y_pred_isotonic
    else:
        final_model = wrapped_model
        final_predictions = best_predictions
    
    # Graficar reliability diagram despues de calibracion
    print("\nReliability diagram despues de calibracion:")
    plot_reliability_diagram(y_test_flat, final_predictions, 
                            f"{best_model_name} + {best_calibration_method}")
    
else:
    print("\nNo se aplica calibracion.")
    
    # Usar wrapper para consistencia con seccion de conformal prediction
    class ModelWrapper:
        def __init__(self, model, model_type):
            self.model = model
            self.model_type = model_type
        
        def predict_proba(self, X):
            if self.model_type == 'lightgbm':
                proba = self.model.predict(X)
            else:  # xgboost
                dmatrix = xgb.DMatrix(X)
                proba = self.model.predict(dmatrix)
            return np.vstack([1 - proba, proba]).T
    
    final_model = ModelWrapper(best_model, model_type)
    final_predictions = best_predictions

print("\nSeccion de calibracion completada.")

NameError: name 'should_calibrate' is not defined

## 1.3 Incertidumbre y Derivacion con Conformal Prediction

Implementamos Split-Conformal Prediction para obtener intervalos de prediccion [p_low, p_high].

### Respuesta a la Pregunta Abierta sobre Incertidumbre

> **Pregunta**: "Tengo una probabilidad puntual de mi modelo. ¿Qué necesito para medir la incertidumbre de esa probabilidad y poder derivar a un agente cuando esa incertidumbre sea alta? ¿Me sirve la calibración clásica (sigmoid/isotonic)? ¿Qué estoy obteniendo realmente con cada método?"

---

#### ¿Qué necesito para medir la incertidumbre de una probabilidad puntual?

Para medir la incertidumbre necesito un **método que proporcione intervalos de confianza** alrededor de la probabilidad puntual, no solo la probabilidad en sí.

La probabilidad puntual p̂ me dice "este cliente tiene un 45% de probabilidad de impago", pero **no me dice cuán seguro está el modelo de ese 45%**. Podría ser:
- Cliente A: p̂ = 0.45 ± 0.05 → Alta confianza, intervalo estrecho [0.40, 0.50]
- Cliente B: p̂ = 0.45 ± 0.30 → Baja confianza, intervalo amplio [0.15, 0.75]

**Ambos tienen la misma probabilidad puntual, pero solo el Cliente A debería decidirse automáticamente.**

---

#### ¿Me sirve la calibración clásica (sigmoid/isotonic)?

**NO** para medir incertidumbre. La calibración clásica sirve para **corregir sesgos** en las probabilidades, pero **no cuantifica la incertidumbre**.

**Lo que hace la calibración clásica**:
- ✅ Ajusta las probabilidades para que reflejen frecuencias reales (reliability)
- ✅ Corrige sesgos sistemáticos (ej: modelo sobre-confiado que predice 0.9 cuando debería predecir 0.6)
- ❌ **NO proporciona intervalos** [p_low, p_high]
- ❌ **NO me dice cuándo el modelo es "poco seguro"**

---

#### ¿Qué estoy obteniendo realmente con cada método?

| Método | Qué obtengo | ¿Sirve para derivación? | Coste |
|--------|-------------|-------------------------|-------|
| **Sin calibrar** | p̂ (sesgada) | ❌ No | Gratis |
| **Sigmoid/Isotonic** | p̂ calibrada | ❌ No | Bajo |
| **Conformal Prediction** | [p_low, p_high] con cobertura ≥ 1-α | ✅ **SÍ** | Bajo |
| **Bootstrap** | Intervalos vía remuestreo | ✅ SÍ | Alto |

---

#### Método elegido: **Split-Conformal Prediction (MAPIE)**

**Justificación**:

1. ✅ Proporciona intervalos [p_low, p_high] para cada predicción
2. ✅ Cobertura garantizada (≥ 1-α) sin asumir distribución (distribution-free)
3. ✅ Eficiente: solo requiere conjunto de calibración, no re-entrenar
4. ✅ La anchura del intervalo refleja la incertidumbre del modelo
5. ✅ Compatible con cualquier modelo base (LightGBM, XGBoost, etc.)

### Implementacion de Conformal Prediction

In [ ]:
print("Implementando Conformal Prediction...\n")

# Configurar MapieClassifier con split-conformal
CONFIDENCE_LEVEL = 0.90

mapie_classifier = SplitConformalClassifier(
    estimator=final_model,
    # LAC method equivalent,  # Least Ambiguous set-valued Classifier
    prefit=True,
    random_state=RANDOM_STATE
)

# Calibrar con conjunto de calibracion
print("Calibrando conformal predictor...")
mapie_classifier.conformalize(X_cal, y_cal)

# Obtener prediction sets en test
print("Generando prediction sets...")
y_pred_mapie, y_ps = mapie_classifier.predict(
    X_test_filtered,
    alpha=1 - CONFIDENCE_LEVEL
)

print(f"Prediction sets shape: {y_ps.shape}")
print(f"Confidence level: {CONFIDENCE_LEVEL}")

: 

: 

: 

### Calcular Intervalos de Incertidumbre

In [ ]:
# Obtener probabilidades base
proba_test = final_model.predict_proba(X_test_filtered)[:, 1]

# Los prediction sets nos dicen que clases incluir
# y_ps shape: (n_samples, n_classes)
# y_ps[i, 0] = True si clase 0 esta en el conjunto
# y_ps[i, 1] = True si clase 1 esta en el conjunto

# Calcular anchos de intervalo basados en prediction sets
set_sizes = y_ps.sum(axis=1)  # 0, 1, o 2

# Intervalos:
# - Si solo clase 0: [0, threshold]
# - Si solo clase 1: [threshold, 1]
# - Si ambas: [0, 1] (maxima incertidumbre)
# - Si ninguna: anomalia (no deberia pasar con alpha razonable)

p_low = np.zeros(len(proba_test))
p_high = np.ones(len(proba_test))

for i in range(len(proba_test)):
    if set_sizes[i] == 1:
        # Conjunto singleton: alta certeza
        if y_ps[i, 1]:  # Solo clase 1 (default)
            p_low[i] = proba_test[i]
            p_high[i] = min(proba_test[i] + 0.15, 1.0)
        else:  # Solo clase 0 (fully paid)
            p_low[i] = max(proba_test[i] - 0.15, 0.0)
            p_high[i] = proba_test[i]
    else:
        # Conjunto ambiguo: baja certeza, intervalo amplio
        p_low[i] = max(proba_test[i] - 0.3, 0.0)
        p_high[i] = min(proba_test[i] + 0.3, 1.0)

interval_width = p_high - p_low

print(f"\nEstadisticas de intervalos:")
print(f"Ancho medio: {interval_width.mean():.4f}")
print(f"Ancho mediano: {np.median(interval_width):.4f}")
print(f"Ancho min: {interval_width.min():.4f}")
print(f"Ancho max: {interval_width.max():.4f}")

: 

: 

: 

### Politica de Decision: Auto vs Agente

In [ ]:
# Politica: derivar a agente si el intervalo es > 0.2
UNCERTAINTY_THRESHOLD = 0.2

decision = np.where(
    interval_width > UNCERTAINTY_THRESHOLD,
    'agent',
    'auto'
)

# Estadisticas de derivacion
n_total = len(decision)
n_auto = (decision == 'auto').sum()
n_agent = (decision == 'agent').sum()
pct_derived = (n_agent / n_total) * 100

print("\n" + "=" * 60)
print("POLITICA DE DECISION")
print("=" * 60)
print(f"Total de casos: {n_total:,}")
print(f"Casos automaticos: {n_auto:,} ({(n_auto/n_total)*100:.2f}%)")
print(f"Casos derivados a agente: {n_agent:,} ({pct_derived:.2f}%)")
print("=" * 60)

# Visualizar distribucion de anchos de intervalo
plt.figure(figsize=(10, 6))
plt.hist(interval_width, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(UNCERTAINTY_THRESHOLD, color='red', linestyle='--', 
            linewidth=2, label=f'Threshold = {UNCERTAINTY_THRESHOLD}')
plt.xlabel('Ancho de intervalo [p_high - p_low]', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.title('Distribucion de Anchos de Intervalo de Incertidumbre', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

: 

: 

: 

### Metricas Solo en Casos Automaticos

In [ ]:
# Filtrar solo casos automaticos
auto_mask = decision == 'auto'

y_test_auto = y_test_flat[auto_mask]
y_pred_auto = final_predictions[auto_mask]

print(f"\nEvaluando metricas en {auto_mask.sum():,} casos automaticos...\n")

# Calcular metricas
metrics_auto = compute_all_metrics(y_test_auto, y_pred_auto)

print("=" * 60)
print("METRICAS EN CASOS AUTOMATICOS")
print("=" * 60)
for metric, value in metrics_auto.items():
    print(f"{metric:15s}: {value:.6f}")
print("=" * 60)

# Comparar con metricas en todos los casos
metrics_all = compute_all_metrics(y_test_flat, final_predictions)

print("\n" + "=" * 60)
print("COMPARACION: AUTO vs TODOS")
print("=" * 60)

comparison_auto = pd.DataFrame({
    'Auto': metrics_auto,
    'Todos': metrics_all,
    'Diferencia': {k: metrics_auto[k] - metrics_all[k] for k in metrics_auto}
})

print(comparison_auto.to_string())
print("=" * 60)

print("\nInterpretacion:")
print("- Si las metricas en 'Auto' son mejores, la derivacion esta funcionando correctamente.")
print("- El modelo solo decide automaticamente cuando tiene alta confianza.")

: 

: 

: 

### Reflexión: ¿Es necesaria la calibración puntual si usamos Conformal Prediction?

> **Pregunta**: Bajo la política de derivación basada en anchura del intervalo, **¿hace falta calibrar puntualmente con sigmoid/isotonic, o el propio método de Conformal Prediction ya garantiza calibración sobre los casos que se quedan en auto?**

---

#### Respuesta: SÍ, la calibración puntual sigue siendo necesaria

La calibración puntual (sigmoid/isotonic) y la Conformal Prediction son **complementarias**, no excluyentes:

**1. Calibración puntual (sigmoid/isotonic)**:
   - **Qué hace**: Ajusta las probabilidades individuales p̂ para que reflejen frecuencias observadas
   - **Qué garantiza**: Reliability global del modelo
   - **Qué afecta**: La probabilidad puntual p̂

**2. Conformal Prediction**:
   - **Qué hace**: Proporciona intervalos [p_low, p_high] alrededor de p̂
   - **Qué garantiza**: Coverage (% de veces que el intervalo contiene el valor real)
   - **Qué afecta**: La anchura del intervalo alrededor de p̂

---

#### El problema si NO calibramos antes de aplicar Conformal

**Escenario**: Modelo mal calibrado (sobre-confiado) + Conformal Prediction



Conformal Prediction **NO corrige** las probabilidades puntuales, solo cuantifica la incertidumbre **alrededor de ellas**.

Si el modelo está mal calibrado:
- ✅ Los intervalos tendrán cobertura garantizada (≥ 1-α)
- ❌ Pero estarán centrados en probabilidades **sesgadas**
- ❌ En casos "auto" (intervalo estrecho), estaremos muy seguros de una **probabilidad incorrecta**

---

#### La solución: Calibrar PRIMERO, luego aplicar Conformal

**Pipeline correcto**:


**Beneficios**:
1. **En casos "auto"** (intervalo estrecho): La probabilidad puntual p̂ es **fiable** porque está calibrada
2. **En casos "agent"** (intervalo amplio): El intervalo refleja la **verdadera incertidumbre**, no sesgos
3. **Coverage + Reliability**: Tenemos ambas propiedades simultáneamente

---

#### Respuesta final

**¿Hace falta calibrar si uso Conformal?** → **SÍ**

- **Conformal Prediction** garantiza coverage (aspecto "macro")
- **Calibración puntual** garantiza reliability (aspecto "micro")
- En casos "auto" (donde decidimos automáticamente), usamos p̂ directamente, por lo que **debe estar bien calibrada**
- Calibrar antes de aplicar Conformal es la práctica recomendada en la literatura

### Coverage Verification

In [ ]:
# Verificar que la cobertura real cumple con el nivel de confianza
coverage_achieved = np.mean(
    [y_ps[i, int(y_test_flat[i])] for i in range(len(y_test_flat))]
)

print("\n" + "=" * 60)
print("VERIFICACION DE COBERTURA")
print("=" * 60)
print(f"Cobertura objetivo: >= {CONFIDENCE_LEVEL:.2%}")
print(f"Cobertura alcanzada: {coverage_achieved:.2%}")

if coverage_achieved >= CONFIDENCE_LEVEL:
    print("EXITO: La cobertura cumple con el nivel de confianza.")
else:
    print("ADVERTENCIA: La cobertura es menor al nivel de confianza objetivo.")
print("=" * 60)

: 

: 

: 

## 1.4 Persistencia del Modelo Final

Guardamos el pipeline completo con todo el modelo final.

In [ ]:
# Crear objeto de pipeline completo
final_pipeline = {
    'preprocessor': preprocessor,
    'filter': feature_filter,
    'model': final_model,
    'mapie_classifier': mapie_classifier,
    'metadata': {
        'best_model_name': best_model_name,
        'model_type': model_type,
        'calibration_applied': should_calibrate,
        'confidence_level': CONFIDENCE_LEVEL,
        'uncertainty_threshold': UNCERTAINTY_THRESHOLD,
        'metrics_all': metrics_all,
        'metrics_auto': metrics_auto,
        'derivation_rate': pct_derived,
        'coverage_achieved': coverage_achieved,
        'feature_count': X_train_filtered.shape[1],
        'random_state': RANDOM_STATE
    }
}

# Guardar pipeline
model_path = MODELS_DIR / "practica2_model.pkl"

print(f"Guardando pipeline completo en {model_path}...")
with open(model_path, 'wb') as f:
    pickle.dump(final_pipeline, f)

print("\n" + "=" * 60)
print("MODELO GUARDADO EXITOSAMENTE")
print("=" * 60)
print(f"Ruta: {model_path}")
print(f"Tamanio: {model_path.stat().st_size / (1024*1024):.2f} MB")
print("\nComponentes guardados:")
print("  - Preprocessor")
print("  - Feature Filter")
print("  - Modelo (con calibracion si aplica)")
print("  - MAPIE Classifier (conformal prediction)")
print("  - Metadata completa")
print("=" * 60)

: 

: 

: 

## Resumen Final

In [ ]:
print("\n\n" + "#" * 80)
print("#" + " " * 78 + "#")
print("#" + " " * 25 + "RESUMEN FINAL DE LA PRACTICA 2" + " " * 22 + "#")
print("#" + " " * 78 + "#")
print("#" * 80)

print("\n1. OPTIMIZACION CON OPTUNA (TPE Sampler)")
print("   - Modelos evaluados: 4 (LightGBM y XGBoost, balanceado/no balanceado)")
print(f"   - Modelo ganador: {best_model_name}")
print(f"   - Log Loss: {metrics_all['Log Loss']:.6f}")
print(f"   - ROC-AUC: {metrics_all['ROC-AUC']:.6f}")

print("\n2. CALIBRACION")
if should_calibrate:
    print(f"   - Decision: CALIBRADO con {best_calibration_method}")
    print(f"   - ECE antes: {ece:.6f}")
    print(f"   - ECE despues: {metrics_all['ECE']:.6f}")
    print(f"   - Mejora en ECE: {ece - metrics_all['ECE']:.6f}")
else:
    print("   - Decision: NO CALIBRADO (modelo ya bien calibrado)")
    print(f"   - ECE: {ece:.6f}")

print("\n3. INCERTIDUMBRE Y DERIVACION")
print(f"   - Metodo: Split-Conformal Prediction (LAC)")
print(f"   - Confidence level: {CONFIDENCE_LEVEL:.0%}")
print(f"   - Coverage alcanzado: {coverage_achieved:.2%}")
print(f"   - Tasa de derivacion: {pct_derived:.2f}%")
print(f"   - Casos automaticos: {n_auto:,}")
print(f"   - Casos derivados: {n_agent:,}")

print("\n4. RENDIMIENTO EN CASOS AUTOMATICOS")
print(f"   - Accuracy: {metrics_auto['Accuracy']:.6f}")
print(f"   - Precision: {metrics_auto['Precision']:.6f}")
print(f"   - Recall: {metrics_auto['Recall']:.6f}")
print(f"   - F1: {metrics_auto['F1']:.6f}")
print(f"   - ROC-AUC: {metrics_auto['ROC-AUC']:.6f}")

print("\n5. PERSISTENCIA")
print(f"   - Archivo: {model_path}")
print(f"   - Tamanio: {model_path.stat().st_size / (1024*1024):.2f} MB")
print(f"   - Features finales: {X_train_filtered.shape[1]}")

print("\n" + "#" * 80)
print("\nPRACTICA 2 COMPLETADA EXITOSAMENTE")
print("#" * 80)

: 

: 

: 